In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (log_loss, roc_auc_score, f1_score, 
                            accuracy_score, precision_score, recall_score, 
                            matthews_corrcoef, confusion_matrix)
import optuna
from sklearn.model_selection import StratifiedKFold

In [2]:
X_train = pd.read_csv('../datasets/preprocessed/X_train2.csv')
y_train = pd.read_csv('../datasets/preprocessed/y_train2.csv').squeeze()  # Series 변환
X_val = pd.read_csv('../datasets/preprocessed/X_val2.csv')
y_val = pd.read_csv('../datasets/preprocessed/y_val2.csv').squeeze()

In [4]:
pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"Positive weight for class 1: {pos_weight:.2f}")

Positive weight for class 1: 1.02


In [5]:
def main_metric(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return (acc + f1 + auc) / 3

In [6]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', pos_weight-1, pos_weight+2),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'eval_metric': 'Recall',  # 재현율 중심 모니터링
        'early_stopping_rounds': 50,
        'verbose': False,
        'random_seed': 42
    }
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=(X_fold_val, y_fold_val),
            cat_features=list(X_train.select_dtypes(include='object').columns),
            use_best_model=True
        )
        
        y_pred = model.predict(X_fold_val)
        score = main_metric(y_fold_val, y_pred)
        cv_scores.append(score)
    
    return np.mean(cv_scores)

In [7]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=3600) 

[I 2025-05-29 02:22:17,612] A new study created in memory with name: no-name-ea198938-822d-448b-9900-b2f84c521bdd
[I 2025-05-29 02:22:18,504] Trial 0 finished with value: 0.3599809097572378 and parameters: {'learning_rate': 0.007321581018686275, 'depth': 6, 'l2_leaf_reg': 0.0704184007442859, 'random_strength': 7.942419450717959, 'border_count': 228, 'scale_pos_weight': 0.3902716079368278, 'grow_policy': 'Depthwise'}. Best is trial 0 with value: 0.3599809097572378.
[I 2025-05-29 02:22:18,945] Trial 1 finished with value: 0.4545741167661304 and parameters: {'learning_rate': 0.0019612713824535832, 'depth': 7, 'l2_leaf_reg': 0.12002180301303346, 'random_strength': 2.5078694774473203, 'border_count': 32, 'scale_pos_weight': 0.5013598011248082, 'grow_policy': 'SymmetricTree'}. Best is trial 1 with value: 0.4545741167661304.
[I 2025-05-29 02:22:27,551] Trial 2 finished with value: 0.5967623038352832 and parameters: {'learning_rate': 0.055770841520487356, 'depth': 5, 'l2_leaf_reg': 0.004533526

In [8]:
best_params = study.best_params
best_params.update({
    'eval_metric': 'Recall',
    'early_stopping_rounds': 50,
    'random_seed': 42
})

final_model = CatBoostClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=list(X_train.select_dtypes(include='object').columns),
    use_best_model=True
)

0:	learn: 0.4209868	test: 0.3931818	best: 0.3931818 (0)	total: 28.3ms	remaining: 28.3s
1:	learn: 0.4704411	test: 0.4236364	best: 0.4236364 (1)	total: 52.1ms	remaining: 26s
2:	learn: 0.4921555	test: 0.4259091	best: 0.4259091 (2)	total: 75.9ms	remaining: 25.2s
3:	learn: 0.5067076	test: 0.4281818	best: 0.4281818 (3)	total: 103ms	remaining: 25.8s
4:	learn: 0.5196680	test: 0.4286364	best: 0.4286364 (4)	total: 129ms	remaining: 25.6s
5:	learn: 0.5293315	test: 0.4368182	best: 0.4368182 (5)	total: 152ms	remaining: 25.3s
6:	learn: 0.5438836	test: 0.4409091	best: 0.4409091 (6)	total: 176ms	remaining: 24.9s
7:	learn: 0.5578672	test: 0.4468182	best: 0.4468182 (7)	total: 201ms	remaining: 24.9s
8:	learn: 0.5601410	test: 0.4522727	best: 0.4522727 (8)	total: 226ms	remaining: 24.8s
9:	learn: 0.5665075	test: 0.4572727	best: 0.4572727 (9)	total: 254ms	remaining: 25.1s
10:	learn: 0.5750341	test: 0.4595455	best: 0.4595455 (10)	total: 279ms	remaining: 25.1s
11:	learn: 0.5825375	test: 0.4595455	best: 0.459545

In [9]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise']),
        'eval_metric': 'Logloss',
        'early_stopping_rounds': 50,
        'verbose': False,
        'random_seed': 42
    }
    
    model = CatBoostClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=list(X_train.select_dtypes(include='object').columns),
        use_best_model=True
    )
    preds_proba = model.predict_proba(X_val)[:, 1]
    return log_loss(y_val, preds_proba)

In [10]:
def recall_optimized_threshold(model, X_val, y_val, min_precision=0.5):
    y_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 100)
    best_thresh = 0.5
    best_recall = 0
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        
        # Recall 중심
        if precision >= min_precision and recall > best_recall:
            best_recall = recall
            best_thresh = thresh
            
    return best_thresh, best_recall

In [11]:
optimal_threshold, best_recall = recall_optimized_threshold(final_model, X_val, y_val, min_precision=0.5)
print(f"Optimal Threshold: {optimal_threshold:.4f}, Best Recall: {best_recall:.4f}")


Optimal Threshold: 0.1000, Best Recall: 0.9195


In [12]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    metrics = {
        'logloss': log_loss(y, y_proba),
        'auc': roc_auc_score(y, y_proba),
        'f1': f1_score(y, y_pred),
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'mcc': matthews_corrcoef(y, y_pred)
    }
    
    metrics['main_metric'] = (metrics['accuracy'] + metrics['f1'] + metrics['auc']) / 3
    return metrics

In [13]:
val_metrics = evaluate_model(final_model, X_val, y_val, optimal_threshold)

In [14]:
print("\n" + "="*60)
print(f"{'Metric':<15} {'Value':<10} {'Improvement Focus':<25}")
print("-"*60)
for metric, value in val_metrics.items():
    focus = ""
    if metric == 'recall': focus = "MAXIMIZE (Core Target)"
    elif metric == 'main_metric': focus = "Primary Optimization Goal"
    elif metric in ['f1', 'auc', 'accuracy']: focus = "Main Metric Component"
    print(f"{metric:<15} {value:.6f}   {focus}")
print("="*60)


Metric          Value      Improvement Focus        
------------------------------------------------------------
logloss         0.769888   
auc             0.701207   Main Metric Component
f1              0.689150   Main Metric Component
accuracy        0.588779   Main Metric Component
precision       0.551076   
recall          0.919545   MAXIMIZE (Core Target)
mcc             0.242223   
main_metric     0.659712   Primary Optimization Goal


In [15]:
y_pred_val = (final_model.predict_proba(X_val)[:, 1] >= optimal_threshold).astype(int)
cm = confusion_matrix(y_val, y_pred_val)
cm_df = pd.DataFrame(cm, 
                   index=['Actual 0', 'Actual 1'],
                   columns=['Predicted 0', 'Predicted 1'])

print("\nConfusion Matrix (Recall-Optimized):")
print(cm_df)
print(f"\nTrue Positives (Correct 1s): {cm[1,1]}/{cm[1].sum()} ({cm[1,1]/cm[1].sum():.2%})")


Confusion Matrix (Recall-Optimized):
          Predicted 0  Predicted 1
Actual 0          590         1648
Actual 1          177         2023

True Positives (Correct 1s): 2023/2200 (91.95%)
